# PMLB People Comparison

This notebook compares three real-data models on a PMLB people-related dataset: CNN only, QNN with linear concatenation, and My Method. It tries `people` first and falls back to `adult` if `people` is not a valid PMLB dataset.

In [3]:
import copy
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from pmlb import classification_dataset_names, fetch_data
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
REQUESTED_DATASET = "people"
FALLBACK_DATASET = "adult"
MAX_ROWS = 5000
EPOCHS = 12
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
N_QUBITS = 4
N_Q_LAYERS = 2
RESULTS_DIR = Path("/home/sammarv/quantum_corrosion/results/pmlb_people_comparison")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 1. Set Up Imports and Configuration

Resolve the requested PMLB dataset name, apply the fallback, and build the shared split configuration.

In [2]:
def resolve_people_dataset() -> str:
    if REQUESTED_DATASET in classification_dataset_names:
        return REQUESTED_DATASET
    if FALLBACK_DATASET in classification_dataset_names:
        return FALLBACK_DATASET
    raise ValueError(f'Neither {REQUESTED_DATASET!r} nor {FALLBACK_DATASET!r} is available in PMLB')


def load_dataset(dataset_name: str) -> pd.DataFrame:
    data = fetch_data(dataset_name, dropna=True)
    if not isinstance(data, pd.DataFrame):
        raise TypeError(f'Expected DataFrame, got {type(data)!r}')
    return data


def maybe_cap_rows(data: pd.DataFrame, cap: int = MAX_ROWS, seed: int = SEED) -> pd.DataFrame:
    if len(data) <= cap:
        return data.copy()
    capped, _ = train_test_split(
        data,
        train_size=cap,
        random_state=seed,
        stratify=data['target'],
    )
    return capped.reset_index(drop=True)


RESOLVED_DATASET = resolve_people_dataset()
raw_df = load_dataset(RESOLVED_DATASET)
raw_df = maybe_cap_rows(raw_df, cap=MAX_ROWS)
print('Requested dataset:', REQUESTED_DATASET)
print('Resolved dataset:', RESOLVED_DATASET)
print('Raw shape:', raw_df.shape)
print('Columns:', raw_df.columns.tolist())
print('Target balance:\n', raw_df['target'].value_counts(dropna=False))

Requested dataset: people
Resolved dataset: adult
Raw shape: (5000, 15)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'target']
Target balance:
 target
1    3804
0    1196
Name: count, dtype: int64


## 2. Define Core Data Structures

Split the real PMLB data, encode categoricals on the train split only, and build the shared loaders.

In [5]:
def split_and_encode(data: pd.DataFrame, seed: int = SEED):
    if 'target' not in data.columns:
        raise ValueError("PMLB dataset must contain a target column")

    train_df, temp_df = train_test_split(
        data,
        test_size=0.30,
        random_state=seed,
        stratify=data['target'],
    )
    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=seed,
        stratify=temp_df['target'],
    )

    feature_cols = [c for c in data.columns if c != 'target']
    categorical_cols = train_df[feature_cols].select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    encoder = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )

    X_train = encoder.fit_transform(train_df[feature_cols])
    X_valid = encoder.transform(valid_df[feature_cols])
    X_test = encoder.transform(test_df[feature_cols])

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(train_df['target'].astype(str))
    y_valid = label_encoder.transform(valid_df['target'].astype(str))
    y_test = label_encoder.transform(test_df['target'].astype(str))

    X_train = np.asarray(X_train, dtype=np.float32)
    X_valid = np.asarray(X_valid, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)

    return {
        'feature_cols': feature_cols,
        'categorical_cols': categorical_cols,
        'numeric_cols': numeric_cols,
        'encoder': encoder,
        'label_encoder': label_encoder,
        'X_train': X_train,
        'X_valid': X_valid,
        'X_test': X_test,
        'y_train': y_train,
        'y_valid': y_valid,
        'y_test': y_test,
    }


prep = split_and_encode(raw_df, seed=SEED)
N_FEATURES = prep['X_train'].shape[1]
N_CLASSES = len(prep['label_encoder'].classes_)
CLASS_NAMES = prep['label_encoder'].classes_.tolist()

print('Encoded feature count:', N_FEATURES)
print('Classes:', CLASS_NAMES)
print('Train/Valid/Test:', prep['X_train'].shape, prep['X_valid'].shape, prep['X_test'].shape)
print('Train class balance:', np.bincount(prep['y_train']))

Encoded feature count: 14
Classes: ['0', '1']
Train/Valid/Test: (3500, 14) (750, 14) (750, 14)
Train class balance: [ 837 2663]


## 3. Implement Primary Functions

Define the CNN-only baseline, the QNN with linear concatenation, and the custom My Method hybrid.

## 4. Handle Validation and Errors

Build the loaders and confirm the model output shapes before training.

In [7]:
train_ds = TensorDataset(torch.tensor(prep['X_train'], dtype=torch.float32), torch.tensor(prep['y_train'], dtype=torch.long))
valid_ds = TensorDataset(torch.tensor(prep['X_valid'], dtype=torch.float32), torch.tensor(prep['y_valid'], dtype=torch.long))
test_ds = TensorDataset(torch.tensor(prep['X_test'], dtype=torch.float32), torch.tensor(prep['y_test'], dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

assert len(train_ds) > 0 and len(valid_ds) > 0 and len(test_ds) > 0
assert prep['X_train'].shape[1] == prep['X_valid'].shape[1] == prep['X_test'].shape[1]

sample_x, sample_y = next(iter(train_loader))
with torch.no_grad():
    cnn_logits = cnn_only_model(sample_x.to(device))
    qnn_logits = qnn_model(sample_x.to(device))
    my_logits = my_method_model(sample_x.to(device))

assert cnn_logits.shape == (sample_x.shape[0], n_classes)
assert qnn_logits.shape == (sample_x.shape[0], n_classes)
assert my_logits.shape == (sample_x.shape[0], n_classes)
print('Validation passed:', sample_x.shape, sample_y.shape)

Validation passed: torch.Size([64, 14]) torch.Size([64])


## 5. Run a Minimal Usage Example

Train the three models on the resolved real PMLB dataset and keep the best validation checkpoint for each one.

In [8]:
def evaluate_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
    return np.array(y_true), np.array(y_pred)


def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def train_model(model, train_loader, valid_loader, epochs=EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, patience=3):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    best_state = copy.deepcopy(model.state_dict())
    best_score = -1.0
    best_epoch = -1
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        train_true = []
        train_pred = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            train_true.extend(yb.cpu().numpy().tolist())
            train_pred.extend(torch.argmax(logits, dim=1).detach().cpu().numpy().tolist())

        train_loss = running_loss / len(train_loader.dataset)
        train_metrics = compute_metrics(train_true, train_pred)
        valid_true, valid_pred = evaluate_model(model, valid_loader)
        valid_metrics = compute_metrics(valid_true, valid_pred)
        history.append({'epoch': epoch, 'train_loss': train_loss, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'valid_{k}': v for k, v in valid_metrics.items()}})
        print(f'Epoch {epoch:02d}/{epochs} train_loss={train_loss:.4f} valid_acc={valid_metrics["accuracy"]:.4f} valid_f1={valid_metrics["macro_f1"]:.4f}')

        if valid_metrics['macro_f1'] > best_score:
            best_score = valid_metrics['macro_f1']
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model, history, {'best_epoch': best_epoch, 'best_macro_f1': best_score}


cnn_only_model, cnn_history, cnn_best = train_model(CNN_ONLY_MODEL := cnn_only_model, train_loader, valid_loader)
qnn_model, qnn_history, qnn_best = train_model(QNN_MODEL := qnn_model, train_loader, valid_loader)
my_method_model, my_history, my_best = train_model(MY_METHOD_MODEL := my_method_model, train_loader, valid_loader)

print('CNN only best:', cnn_best)
print('QNN linear concat best:', qnn_best)
print('My Method best:', my_best)

Epoch 01/12 train_loss=0.5747 valid_acc=0.7613 valid_f1=0.4322
Epoch 02/12 train_loss=0.5049 valid_acc=0.7747 valid_f1=0.4975
Epoch 03/12 train_loss=0.4892 valid_acc=0.7867 valid_f1=0.5498
Epoch 04/12 train_loss=0.4819 valid_acc=0.7893 valid_f1=0.5660
Epoch 05/12 train_loss=0.4795 valid_acc=0.7987 valid_f1=0.6023
Epoch 06/12 train_loss=0.4732 valid_acc=0.7933 valid_f1=0.5856
Epoch 07/12 train_loss=0.4705 valid_acc=0.7973 valid_f1=0.6012
Epoch 08/12 train_loss=0.4649 valid_acc=0.7960 valid_f1=0.6113
Epoch 09/12 train_loss=0.4601 valid_acc=0.7840 valid_f1=0.5442
Epoch 10/12 train_loss=0.4548 valid_acc=0.7907 valid_f1=0.5771
Epoch 11/12 train_loss=0.4519 valid_acc=0.8027 valid_f1=0.6227
Epoch 12/12 train_loss=0.4418 valid_acc=0.8040 valid_f1=0.6549
Epoch 01/12 train_loss=0.5050 valid_acc=0.8040 valid_f1=0.6100
Epoch 02/12 train_loss=0.3942 valid_acc=0.8333 valid_f1=0.7307
Epoch 03/12 train_loss=0.3694 valid_acc=0.8347 valid_f1=0.7321
Epoch 04/12 train_loss=0.3566 valid_acc=0.8373 valid_f1

## 6. Add Basic Tests

Evaluate the best checkpoint for each model on the held-out test split and save the outputs.

In [9]:
cnn_test_true, cnn_test_pred = evaluate_model(cnn_only_model, test_loader)
qnn_test_true, qnn_test_pred = evaluate_model(qnn_model, test_loader)
my_test_true, my_test_pred = evaluate_model(my_method_model, test_loader)

cnn_metrics = compute_metrics(cnn_test_true, cnn_test_pred)
qnn_metrics = compute_metrics(qnn_test_true, qnn_test_pred)
my_metrics = compute_metrics(my_test_true, my_test_pred)

print('\nCNN only report')
print(cnn_metrics['classification_report'])
print('\nQNN with Linear Concatenation report')
print(qnn_metrics['classification_report'])
print('\nMy Method report')
print(my_metrics['classification_report'])

cnn_predictions = pd.DataFrame({'y_true': cnn_test_true, 'y_pred': cnn_test_pred, 'model': 'cnn_only'})
qnn_predictions = pd.DataFrame({'y_true': qnn_test_true, 'y_pred': qnn_test_pred, 'model': 'qnn_linear_concat'})
my_predictions = pd.DataFrame({'y_true': my_test_true, 'y_pred': my_test_pred, 'model': 'my_method'})

cnn_model_path = RESULTS_DIR / 'cnn_only.pt'
qnn_model_path = RESULTS_DIR / 'qnn_linear_concat.pt'
my_model_path = RESULTS_DIR / 'my_method.pt'
cnn_predictions_path = RESULTS_DIR / 'cnn_only_predictions.csv'
qnn_predictions_path = RESULTS_DIR / 'qnn_linear_concat_predictions.csv'
my_predictions_path = RESULTS_DIR / 'my_method_predictions.csv'
metrics_path = RESULTS_DIR / 'metrics.json'

torch.save(cnn_only_model.state_dict(), cnn_model_path)
torch.save(qnn_model.state_dict(), qnn_model_path)
torch.save(my_method_model.state_dict(), my_model_path)

cnn_predictions.to_csv(cnn_predictions_path, index=False)
qnn_predictions.to_csv(qnn_predictions_path, index=False)
my_predictions.to_csv(my_predictions_path, index=False)

metrics_payload = {
    'requested_dataset': REQUESTED_DATASET,
    'resolved_dataset': RESOLVED_DATASET,
    'sample_cap': MAX_ROWS,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'n_features': n_features,
    'n_classes': n_classes,
    'class_names': CLASS_NAMES,
    'cnn_best': cnn_best,
    'qnn_best': qnn_best,
    'my_best': my_best,
    'cnn_metrics': cnn_metrics,
    'qnn_metrics': qnn_metrics,
    'my_metrics': my_metrics,
}
with open(metrics_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)

print('Saved CNN only model:', cnn_model_path)
print('Saved QNN model:', qnn_model_path)
print('Saved My Method model:', my_model_path)
print('Saved metrics:', metrics_path)


CNN only report
              precision    recall  f1-score   support

           0       0.68      0.26      0.38       180
           1       0.80      0.96      0.88       570

    accuracy                           0.79       750
   macro avg       0.74      0.61      0.63       750
weighted avg       0.78      0.79      0.76       750


QNN with Linear Concatenation report
              precision    recall  f1-score   support

           0       0.71      0.47      0.57       180
           1       0.85      0.94      0.89       570

    accuracy                           0.83       750
   macro avg       0.78      0.71      0.73       750
weighted avg       0.82      0.83      0.81       750


My Method report
              precision    recall  f1-score   support

           0       0.65      0.61      0.63       180
           1       0.88      0.89      0.89       570

    accuracy                           0.83       750
   macro avg       0.76      0.75      0.76       750
w

In [6]:
n_features = prep['X_train'].shape[1]
n_classes = len(CLASS_NAMES)
n_qubits = min(N_QUBITS, n_features)


def make_quantum_device(n_q):
    return qml.device('default.qubit', wires=n_q)


qdev = make_quantum_device(n_qubits)


@qml.qnode(qdev, interface='torch', diff_method='backprop')
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class CNNOnly(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Unflatten(1, (1, input_dim)),
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.net(x)


class QNNLinearConcat(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, q_dim: int, q_layers: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, q_dim),
            nn.Sigmoid(),
        )
        self.q_weights = nn.Parameter(torch.randn(q_layers, q_dim) * 0.1)
        self.classical_head = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
        )
        self.fusion = nn.Linear(64 + q_dim, output_dim)

    def forward(self, x):
        q_inputs = self.encoder(x) * math.pi
        q_outputs = []
        for sample in q_inputs:
            out = qnode(sample, self.q_weights)
            if isinstance(out, list):
                out = torch.stack([val if isinstance(val, torch.Tensor) else torch.as_tensor(val, dtype=sample.dtype, device=sample.device) for val in out])
            if not isinstance(out, torch.Tensor):
                out = torch.as_tensor(out, dtype=sample.dtype, device=sample.device)
            q_outputs.append(out.float())
        q_outputs = torch.stack(q_outputs, dim=0)
        c_outputs = self.classical_head(x)
        fused = torch.cat([c_outputs, q_outputs], dim=1)
        return self.fusion(fused)


class MyMethod(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, q_dim: int, q_layers: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.q_proj = nn.Sequential(nn.Linear(64, q_dim), nn.Sigmoid())
        self.q_weights = nn.Parameter(torch.randn(q_layers, q_dim) * 0.1)
        self.film = nn.Sequential(nn.Linear(q_dim, 64), nn.GELU(), nn.Linear(64, 128))
        self.gate = nn.Sequential(nn.Linear(64 + q_dim, 64), nn.Sigmoid())
        self.head = nn.Linear(64, output_dim)

    def forward(self, x):
        c_feat = self.backbone(x)
        q_inputs = self.q_proj(c_feat) * math.pi
        q_outputs = []
        for sample in q_inputs:
            out = qnode(sample, self.q_weights)
            if isinstance(out, list):
                out = torch.stack([val if isinstance(val, torch.Tensor) else torch.as_tensor(val, dtype=sample.dtype, device=sample.device) for val in out])
            if not isinstance(out, torch.Tensor):
                out = torch.as_tensor(out, dtype=sample.dtype, device=sample.device)
            q_outputs.append(out.float())
        q_outputs = torch.stack(q_outputs, dim=0)
        film_params = self.film(q_outputs)
        gamma, beta = torch.chunk(film_params, 2, dim=1)
        c_mod = c_feat * (1.0 + gamma) + beta
        gate = self.gate(torch.cat([c_mod, q_outputs], dim=1))
        fused = gate * c_mod + (1.0 - gate) * c_feat
        return self.head(fused)


def make_classification_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


cnn_only_model = CNNOnly(n_features, n_classes).to(device)
qnn_model = QNNLinearConcat(n_features, n_classes, n_qubits, N_Q_LAYERS).to(device)
my_method_model = MyMethod(n_features, n_classes, n_qubits, N_Q_LAYERS).to(device)

print('CNN only params:', sum(p.numel() for p in cnn_only_model.parameters() if p.requires_grad))
print('QNN linear concat params:', sum(p.numel() for p in qnn_model.parameters() if p.requires_grad))
print('My Method params:', sum(p.numel() for p in my_method_model.parameters() if p.requires_grad))

CNN only params: 1698
QNN linear concat params: 2326
My Method params: 23630
